In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
!pip install optuna

In [ ]:
!pip install xgboost

In [ ]:
from sklearn.model_selection import train_test_split,StratifiedKFold,cross_val_score
from sklearn.preprocessing import StandardScaler,RobustScaler,LabelEncoder
from sklearn.metrics import (accuracy_score,precision_score,recall_score,f1_score,roc_auc_score,confusion_matrix,classification_report,roc_curve,precision_recall_curve)

import xgboost as xgb
import optuna
# import shap
import warnings
warnings.filterwarnings('ignore')

In [ ]:
df=pd.read_csv("urban_street_food_vendor_survival_dataset.csv")


In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.describe()

Check target variable distribution

In [ ]:
print("Target Variable Distribution:")
print("-"*50)

survival_count=df['vendor_survived'].value_counts()
print(survival_count)

print(f"Survival Rate : {survival_count[1]/len(df)*100:.2f}%")

In [ ]:
from matplotlib import colors
fig,axes=plt.subplots(1,2,figsize=(12,5))

#Bar Chart

survival_count.plot(kind='bar',ax=axes[0],color=['red','green'])
axes[0].set_title('Vendor Survival Distribution')
axes[0].set_xlabel('Survived (0=NO , 1=YES)')
axes[0].set_ylabel('Count')
axes[0].set_xticklabels(["Did Not Survive","Survived"],rotation=0)

#Pie Chart

axes[1].pie(survival_count,labels=["Did not Survive","Survived"],colors=["red","green"],startangle=90)

axes[1].set_title("Survival Percentage")

plt.tight_layout()
plt.show()


EDA :

In [ ]:
df.isnull().sum()

In [ ]:
missing_data=df.isnull().sum()
missing_percentage=(missing_data/len(df))*100

missing_df=pd.DataFrame({
    'Missing Count': missing_data,
    'Missing Percentage': missing_percentage
}).sort_values('Missing Percentage',ascending=False)

print("Missing Value Analysis :")
print(missing_df[missing_df['Missing Count']>0])

For Numerical Columns :

In [ ]:
num_cols=[
    'vendor_age_years','years_in_business','avg_daily_revenue_inr','avg_daily_customers','monthly_stall_rent_inr','hours_open_per_day','competition_within_100m','customer_complaint_rate'
]

for col in num_cols:
    df[col] = df[col].fillna(df[col].median())

For Categorical Columns :

In [ ]:
cat_cols = ['license_status']
df[cat_cols] = df[cat_cols].apply(
    lambda x: x.fillna(x.mode().iloc[0] if not x.mode().empty else "Unknown")
)

In [ ]:
df['monthly_health_inspection_score'] = df['monthly_health_inspection_score'].fillna(
    df['monthly_health_inspection_score'].median()
)

In [ ]:
df['had_fine_last_year'] = df['had_fine_last_year'].fillna(0)

In [ ]:
df.isnull().sum()

In [ ]:
numerical_cols = ['vendor_age_years', 'years_in_business', 'avg_daily_revenue_inr',
                  'avg_daily_customers', 'monthly_stall_rent_inr', 'num_helpers',
                  'hours_open_per_day', 'competition_within_100m',
                  'monthly_health_inspection_score', 'customer_complaint_rate']


In [ ]:
categorical_cols = ['city', 'zone_type', 'food_category', 'license_status',
                    'season_of_observation', 'has_online_presence', 'had_fine_last_year']

In [ ]:
numerical_df=df[numerical_cols + ['vendor_survived']].copy()

numerical_df=numerical_df.fillna(numerical_df.median())

corr_matrix=numerical_df.corr()

target_corr=corr_matrix['vendor_survived'].sort_values(ascending=False)
print("\nTop Correlations with Survival :")
print(target_corr)

Data Cleaning And preprocessing :

In [ ]:
def clean_data(df):
  df_clean=df.copy()

  if 'vendor_id' in df_clean.columns:
    df_clean=df_clean.drop('vendor_id',axis=1)

  df_clean['vendor_age_years']=df_clean['vendor_age_years'].replace(0,np.nan)

  df_clean['had_fine_last_year']=df_clean['had_fine_last_year'].fillna(0).astype(int)
  df_clean['has_online_presence']=df_clean['has_online_presence'].fillna(0).astype(int)

  for col in numerical_cols:
    if col in df_clean.columns:

      upper=df_clean[col].quantile(0.99)
      df_clean[col]=df_clean[col].clip(upper=upper)

      lower=df_clean[col].quantile(0.01)
      df_clean[col]=df_clean[col].clip(lower=lower)

    return df_clean

df_clean=clean_data(df)
print("Data Cleaning Completed !!")
print(f"Original shape : {df.shape}")
print(f"Cleaned shape: {df_clean.shape}")

In [ ]:
df.isnull().sum()

Feature Engineering :

In [ ]:
def create_features(df):
  df_feat=df.copy()

  df_feat['revenue_per_customer']=df_feat['avg_daily_revenue_inr'] / (df_feat['avg_daily_customers']+1)
  df_feat['revenue_per_helper']=df_feat['avg_daily_revenue_inr'] / (df_feat['num_helpers']+1)
  df_feat['customers_per_helper']=df_feat['avg_daily_customers'] / (df_feat['num_helpers']+1)

  df_feat['customers_per_hour']=df_feat['avg_daily_customers'] / (df_feat['hours_open_per_day'] + 1)
  df_feat['revenue_per_hour']=df_feat['avg_daily_revenue_inr'] / (df_feat['hours_open_per_day'] + 1)

  df_feat['age_at_start']=df_feat['vendor_age_years'] - df_feat['years_in_business']
  df_feat['experience_ratio']=df_feat['years_in_business'] / (df_feat['vendor_age_years']+1)

  df_feat['competition_per_customer']=df_feat['competition_within_100m'] / (df_feat['avg_daily_customers']+1)
  df_feat['market_saturation']=df_feat['competition_within_100m'] * df_feat['hours_open_per_day']

  df_feat['monthly_revenue']=df_feat['avg_daily_revenue_inr'] * 30
  df_feat['profit_estimate']=df_feat['monthly_revenue'] - df_feat['monthly_stall_rent_inr']
  df_feat['profit_margin']=df_feat['profit_estimate'] / (df_feat['monthly_revenue'] + 1)

  df_feat['health_score_normalized']=df_feat['monthly_health_inspection_score'] / 100
  df_feat['compliance_risk']=df_feat['had_fine_last_year'] * (1 - df_feat['health_score_normalized'])

  df_feat['satisfaction_score']=1-df_feat['customer_complaint_rate']
  df_feat['revenue_per_complaint']=df_feat['avg_daily_revenue_inr'] / (df_feat['customer_complaint_rate']+ 0.01)

  df_feat['rent_to_revenue_ratio']=df_feat['monthly_stall_rent_inr'] / (df_feat['monthly_revenue']+1)

  return df_feat

df_features=create_features(df_clean)
print(f"Features Created ! New Shape :{df_features.shape}")
print(f"New Features added : {len(df_features.columns)-len(df_clean.columns)}")

In [ ]:
!pip uninstall -y numpy scikit-learn scipy

In [ ]:
!pip install numpy==1.26.4
!pip install scipy==1.11.4
!pip install scikit-learn==1.3.2

Prepare Data for Modeling :

In [ ]:
x=df_features.drop('vendor_survived',axis=1)
y=df_features['vendor_survived']

print("Missing values before imputation:")
print(x.isnull().sum().sort_values(ascending=False).head(10))

#Use KNN imputer for numerical features
from sklearn.impute import KNNImputer

numerical_cols_list=x.select_dtypes(include=[np.number]).columns.tolist()

imputer=KNNImputer(n_neighbors=5)
x_imputed=pd.DataFrame(
    imputer.fit_transform(x[numerical_cols_list]),
    columns=numerical_cols_list,
    index=x.index
)

categorical_cols_list=x.select_dtypes(include=['object']).columns.tolist()
if categorical_cols_list:
  x_final=pd.concat([x_imputed,x[categorical_cols_list]],axis=1)
else:
  x_final=x_imputed

print(f"Missing values after imputation : {x_final.isnull().sum().sum()}")

Encoding Categorical Variables

In [ ]:
from pandas.core.arrays import categorical
from sklearn.preprocessing import LabelEncoder

categorical_cols_final=x_final.select_dtypes(include=['object']).columns.tolist()
print(f"Categorical columns to encode: {categorical_cols_final}")

label_encoders={}
for col in categorical_cols_final:
  le=LabelEncoder()
  x_final[col]=x_final[col].fillna('missing').astype(str)
  x_final[col]=le.fit_transform(x_final[col])
  label_encoders[col]=le

  print(f"Encoded {col} : {len(le.classes_)} unique values")

print(f"final feature shape : {x_final.shape}")

In [ ]:
from sklearn.model_selection import train_test_split

x_temp,x_test,y_temp,y_test=train_test_split(
    x_final,y,test_size=0.2 , random_state=42, stratify=y
)

x_train,x_val,y_train,y_val=train_test_split(
    x_temp,y_temp,test_size=0.25,random_state=42,stratify=y_temp
)

print(f"Training set : {x_train.shape[0]} samples")
print(f"Validation set : {x_val.shape[0]} samples")
print(f"Test set: {x_test.shape[0]} samples")
print(f"\nTarget distribution in training : ")
print(y_train.value_counts(normalize=True))

In [ ]:
from pandas.core.tools.datetimes import Scalar
from sklearn.preprocessing import RobustScaler

scaler=RobustScaler()

x_train_scaled=pd.DataFrame(
    scaler.fit_transform(x_train),
    columns=x_train.columns,
    index=x_train.index
)

x_val_scaled=pd.DataFrame(
    scaler.transform(x_val),
    columns=x_val.columns,
    index=x_val.index
)

x_test_scaled = pd.DataFrame(
    scaler.transform(x_test),
    columns=x_test.columns,
    index=x_test.index
)


print("Feature Scalling Completed !!")
print(f"Feature Ranges after Scaling:")
print(f"Min: {x_train_scaled.min().min():.2f}, Max: {x_train_scaled.max().max():.2f}")

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

class_weights=compute_class_weight(
    'balanced',
    classes=np.unique(y_train),
    y=y_train
)

weight_dict=dict(zip(np.unique(y_train), class_weights))
print(f"Class Weights: {weight_dict}")

sample_weights=np.array([weight_dict[y] for y in y_train])

print(f"Sample weights status :")
print(f"Min: {sample_weights.min():.2f}, Max: {sample_weights.max():.2f}")
print(f"Mean: {sample_weights.mean():.2f}")

In [ ]:
import xgboost
print(xgboost.__file__)

In [ ]:
# !pip install xgboost==1.7.6

In [ ]:
# import sys
# print(sys.executable)

In [ ]:
pip show xgboost

Train XGBoost Model with optuna Optimization

In [ ]:
from xgboost.callback import EarlyStopping

def objective(trial):

  params={
      'n_estimators': trial.suggest_int('n_estimators', 300, 1500, step=100),
      'max_depth':trial.suggest_int('max_depth',3,10),
      'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
      'subsample':trial.suggest_float('subsample',0.6,1.0),
      'colsample_bytree':trial.suggest_float('colsample_bytree',0.6,1.0),
      'colsample_bylevel':trial.suggest_float('colsample_bylevel',0.6,1.0),
      'min_child_weight':trial.suggest_int('min_child_weight',1,10),
      'gamma':trial.suggest_float('gamma',0,0.5),
      'reg_alpha':trial.suggest_float('reg_alpha',1e-8,10.0,log=True),
      'reg_lambda':trial.suggest_float('reg_lambda',1e-8,10.0,log=True),
      'scale_pos_weight':trial.suggest_float('scale_pos_weight',0.5,5.0)
  }

  model=xgb.XGBClassifier(
      **params,
      objective='binary:logistic',
      eval_metric='auc',
      use_label_encoder=False,
      random_state=42,
      n_jobs=-1
  )

  model.fit(
    x_train_scaled, y_train,
    sample_weight=sample_weights,
    eval_set=[(x_val_scaled, y_val)],
    early_stopping_rounds=20,
    verbose=False

)

  y_pred_proba=model.predict_proba(x_val_scaled)[:,1]
  auc=roc_auc_score(y_val,y_pred_proba)

  return auc

print("Starting Hyperparameter optimization with Optuna...")
print("This may take a few minutes...")

study=optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)

study.optimize(objective,n_trials=30,show_progress_bar=True)

print("\n" + "="*50)
print("Best trial results:")
print("="*50)
print(f"Best AUC : {study.best_value:.4f}")
print(f"Best Parameter : {study.best_params}")

best_params=study.best_params

In [ ]:
print("Training final model with best parmeters...")

best_params.update({
    'objective':'binary:logistic',
    'eval_metric':'auc',
    'use_label_encoder':False,
    'random_state':42,
    'n_jobs':-1
})

final_model=xgb.XGBClassifier(**best_params)

final_model.fit(
    x_train_scaled,y_train,
    sample_weight=sample_weights,
    eval_set=[(x_val_scaled,y_val)],
    early_stopping_rounds=20,
    verbose=True
)

print("\n Model Training Completed !!")

Model Evaluation

In [ ]:
y_pred=final_model.predict(x_test_scaled)
y_pred_proba=final_model.predict_proba(x_test_scaled)[:,1]

metrics={
    'Accuracy':accuracy_score(y_test,y_pred),
    'Precesion':precision_score(y_test,y_pred),
    'Recall':recall_score(y_test,y_pred),
    'F1-Score':f1_score(y_test,y_pred),
    'AUC-ROC':roc_auc_score(y_test,y_pred_proba)
}

print("Model Performance Metrics:")
print("-" * 40)
for metric,value in metrics.items():
  print(f"{metric:12s}: {value:.4f}")

print("\n Detailed Classification Report :")
print("-" * 40)
print(classification_report(y_test,y_pred,target_names=['Did Not Survive','Survived']))

In [ ]:
# Cell 21: Visualize model performance
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# 1. Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0,0],
            xticklabels=['Not Survived', 'Survived'],
            yticklabels=['Not Survived', 'Survived'])
axes[0,0].set_title('Confusion Matrix')
axes[0,0].set_xlabel('Predicted')
axes[0,0].set_ylabel('Actual')

# 2. ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[0,1].plot(fpr, tpr, 'b-', linewidth=2, label=f'AUC = {metrics["AUC-ROC"]:.3f}')
axes[0,1].plot([0, 1], [0, 1], 'r--', linewidth=1)
axes[0,1].set_xlabel('False Positive Rate')
axes[0,1].set_ylabel('True Positive Rate')
axes[0,1].set_title('ROC Curve')
axes[0,1].legend()
axes[0,1].grid(True, alpha=0.3)

# 3. Precision-Recall Curve
precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
axes[1,0].plot(recall, precision, 'g-', linewidth=2)
axes[1,0].set_xlabel('Recall')
axes[1,0].set_ylabel('Precision')
axes[1,0].set_title('Precision-Recall Curve')
axes[1,0].grid(True, alpha=0.3)

# 4. Feature Importance
feature_importance = pd.DataFrame({
    'feature': x_train.columns,
    'importance': final_model.feature_importances_
}).sort_values('importance', ascending=True).tail(15)

axes[1,1].barh(range(len(feature_importance)), feature_importance['importance'])
axes[1,1].set_yticks(range(len(feature_importance)))
axes[1,1].set_yticklabels(feature_importance['feature'])
axes[1,1].set_xlabel('Feature Importance')
axes[1,1].set_title('Top 15 Feature Importances')

plt.tight_layout()
plt.show()

In [ ]:
precision,recall,thresholds=precision_recall_curve(y_test,y_pred_proba)

f1_scores=2 * (precision * recall) / (precision + recall + 1e-10)

optimal_idx=np.argmax(f1_scores[:-1])
optimal_threshold=thresholds[optimal_idx]
optimal_f1 = f1_scores[optimal_idx]

print(f"Optimal Classification Threshold: {optimal_threshold:.3f}")
print(f"Optimal F1 Score: {optimal_f1:.3f}")

y_pred_optimal=(y_pred_proba >= optimal_threshold).astype(int)

print("\nMetrics with optimal Threshold:")
print("-"*40)
print(f"Accuracy: {accuracy_score(y_test,y_pred_optimal):.4f}")
print(f"Precision: {precision_score(y_test,y_pred_optimal):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_optimal):.4f}")
# print(f"F1 Score: {f1_score(y_test, y_pred_optimal):.4f}")

In [ ]:
# """
# save_model_for_deployment.py
# Run this script in VS Code after training your model
# """

# import joblib
# import json
# import os
# import pandas as pd
# import numpy as np
# from datetime import datetime

# def save_model_for_streamlit(model, scaler, label_encoders, feature_names, optimal_threshold):
#     """
#     Save model specifically for Streamlit deployment
    
#     Parameters:
#     - model: trained XGBoost model
#     - scaler: fitted scaler
#     - label_encoders: dictionary of LabelEncoders
#     - feature_names: list of feature names
#     - optimal_threshold: float (optimal classification threshold)
#     """
    
#     # Create deployment directory
#     os.makedirs('deployment', exist_ok=True)
#     os.makedirs('deployment/model', exist_ok=True)
    
#     print("="*60)
#     print("SAVING MODEL FOR STREAMLIT DEPLOYMENT")
#     print("="*60)
    
#     # 1. Save model
#     joblib.dump(model, 'deployment/model/xgboost_model.pkl')
#     print("✓ Model saved: deployment/model/xgboost_model.pkl")
    
#     # 2. Save scaler
#     joblib.dump(scaler, 'deployment/model/scaler.pkl')
#     print("✓ Scaler saved: deployment/model/scaler.pkl")
    
#     # 3. Save label encoders
#     joblib.dump(label_encoders, 'deployment/model/label_encoders.pkl')
#     print("✓ Label encoders saved: deployment/model/label_encoders.pkl")
    
#     # 4. Save feature names
#     with open('deployment/model/feature_names.json', 'w') as f:
#         json.dump(feature_names, f, indent=4)
#     print("✓ Feature names saved: deployment/model/feature_names.json")
    
#     # 5. Save optimal threshold
#     with open('deployment/model/optimal_threshold.json', 'w') as f:
#         json.dump({'threshold': float(optimal_threshold)}, f, indent=4)
#     print("✓ Optimal threshold saved: deployment/model/optimal_threshold.json")
    
#     # 6. Save feature engineering function as a separate file
#     feature_engineering_code = '''
# import pandas as pd
# import numpy as np

# def create_features(df):
#     """Create advanced features for prediction"""
#     df_feat = df.copy()
    
#     # Business efficiency metrics
#     df_feat['revenue_per_customer'] = df_feat['avg_daily_revenue_inr'] / (df_feat['avg_daily_customers'] + 1)
#     df_feat['revenue_per_helper'] = df_feat['avg_daily_revenue_inr'] / (df_feat['num_helpers'] + 1)
#     df_feat['customers_per_helper'] = df_feat['avg_daily_customers'] / (df_feat['num_helpers'] + 1)
#     df_feat['customers_per_hour'] = df_feat['avg_daily_customers'] / (df_feat['hours_open_per_day'] + 1)
#     df_feat['revenue_per_hour'] = df_feat['avg_daily_revenue_inr'] / (df_feat['hours_open_per_day'] + 1)
    
#     # Experience metrics
#     df_feat['age_at_start'] = df_feat['vendor_age_years'] - df_feat['years_in_business']
#     df_feat['experience_ratio'] = df_feat['years_in_business'] / (df_feat['vendor_age_years'] + 1)
    
#     # Competition impact
#     df_feat['competition_per_customer'] = df_feat['competition_within_100m'] / (df_feat['avg_daily_customers'] + 1)
#     df_feat['market_saturation'] = df_feat['competition_within_100m'] * df_feat['hours_open_per_day']
    
#     # Profitability
#     df_feat['monthly_revenue'] = df_feat['avg_daily_revenue_inr'] * 30
#     df_feat['profit_estimate'] = df_feat['monthly_revenue'] - df_feat['monthly_stall_rent_inr']
#     df_feat['profit_margin'] = df_feat['profit_estimate'] / (df_feat['monthly_revenue'] + 1)
    
#     # Health and satisfaction
#     df_feat['health_score_normalized'] = df_feat['monthly_health_inspection_score'] / 100
#     df_feat['satisfaction_score'] = 1 - df_feat['customer_complaint_rate']
#     df_feat['rent_to_revenue_ratio'] = df_feat['monthly_stall_rent_inr'] / (df_feat['monthly_revenue'] + 1)
    
#     return df_feat
# '''
    
#     with open('deployment/feature_engineering.py', 'w') as f:
#         f.write(feature_engineering_code)
#     print("✓ Feature engineering saved: deployment/feature_engineering.py")
    
#     # 7. Save model info
#     model_info = {
#         'model_type': 'XGBoost Classifier',
#         'save_date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
#         'n_features': len(feature_names),
#         'optimal_threshold': float(optimal_threshold),
#         'feature_names': feature_names[:10],  # First 10 features
#         'categorical_features': list(label_encoders.keys())
#     }
    
#     with open('deployment/model_info.json', 'w') as f:
#         json.dump(model_info, f, indent=4)
#     print("✓ Model info saved: deployment/model_info.json")
    
#     # 8. Create requirements file
#     requirements = '''
# streamlit==1.28.0
# pandas==2.0.3
# numpy==1.24.3
# scikit-learn==1.3.0
# xgboost==1.7.6
# plotly==5.17.0
# joblib==1.3.1
#     '''
    
#     with open('deployment/requirements.txt', 'w') as f:
#         f.write(requirements.strip())
#     print("✓ Requirements saved: deployment/requirements.txt")
    
#     # 9. Create a test script to verify saved model
#     test_script = '''
# import joblib
# import json
# import pandas as pd
# import numpy as np
# from feature_engineering import create_features

# def test_saved_model():
#     """Test if saved model works correctly"""
    
#     # Load model components
#     model = joblib.load('model/xgboost_model.pkl')
#     scaler = joblib.load('model/scaler.pkl')
#     label_encoders = joblib.load('model/label_encoders.pkl')
    
#     with open('model/feature_names.json', 'r') as f:
#         feature_names = json.load(f)
    
#     with open('model/optimal_threshold.json', 'r') as f:
#         threshold = json.load(f)['threshold']
    
#     # Test with sample data
#     test_data = pd.DataFrame({
#         'city': ['Delhi'],
#         'zone_type': ['Commercial'],
#         'food_category': ['Fast Food'],
#         'license_status': ['Licensed'],
#         'vendor_age_years': [35],
#         'years_in_business': [5],
#         'avg_daily_revenue_inr': [2500],
#         'avg_daily_customers': [120],
#         'monthly_stall_rent_inr': [5000],
#         'num_helpers': [2],
#         'hours_open_per_day': [10],
#         'competition_within_100m': [3],
#         'monthly_health_inspection_score': [85],
#         'had_fine_last_year': [0],
#         'avg_monthly_rainfall_mm': [100],
#         'season_of_observation': ['Summer'],
#         'has_online_presence': [1],
#         'customer_complaint_rate': [0.05]
#     })
    
#     # Create features
#     test_data = create_features(test_data)
    
#     # Encode categorical
#     for col, encoder in label_encoders.items():
#         if col in test_data.columns:
#             test_data[col] = test_data[col].astype(str)
#             test_data[col] = test_data[col].map(
#                 lambda x: encoder.transform([x])[0] if x in encoder.classes_ else -1
#             )
    
#     # Ensure all features exist
#     for col in feature_names:
#         if col not in test_data.columns:
#             test_data[col] = 0
    
#     # Scale and predict
#     X_test = test_data[feature_names]
#     X_test_scaled = scaler.transform(X_test)
    
#     probability = model.predict_proba(X_test_scaled)[0][1]
#     prediction = "Will Survive" if probability >= threshold else "May Not Survive"
    
#     print(f"Test Prediction: {prediction}")
#     print(f"Survival Probability: {probability:.1%}")
#     print("✅ Model test successful!")

# if __name__ == "__main__":
#     test_saved_model()
# '''
    
#     with open('deployment/test_model.py', 'w') as f:
#         f.write(test_script)
#     print("✓ Test script saved: deployment/test_model.py")
    
#     # 10. Create a zip file for easy download
#     import zipfile
#     zip_path = 'street_food_model_for_streamlit.zip'
#     with zipfile.ZipFile(zip_path, 'w') as zipf:
#         for root, dirs, files in os.walk('deployment'):
#             for file in files:
#                 file_path = os.path.join(root, file)
#                 arcname = os.path.relpath(file_path, 'deployment')
#                 zipf.write(file_path, arcname)
    
#     print(f"\n✓ All files zipped: {zip_path}")
    
#     print("\n" + "="*60)
#     print("✅ MODEL SAVED SUCCESSFULLY FOR STREAMLIT!")
#     print("="*60)
#     print("\n📁 Deployment folder structure:")
#     print("deployment/")
#     print("├── model/")
#     print("│   ├── xgboost_model.pkl")
#     print("│   ├── scaler.pkl")
#     print("│   ├── label_encoders.pkl")
#     print("│   ├── feature_names.json")
#     print("│   └── optimal_threshold.json")
#     print("├── feature_engineering.py")
#     print("├── model_info.json")
#     print("├── requirements.txt")
#     print("└── test_model.py")
#     print(f"\n📦 Zip file: {zip_path}")
#     print("\n🚀 Next: Copy the 'deployment' folder to your Streamlit project")
    
#     return zip_path

# # ============================================
# # AFTER TRAINING YOUR MODEL, RUN THIS:
# # ============================================

# # Assuming you have these variables from your training:
# # final_model, scaler, label_encoders, feature_names, optimal_threshold

# # Uncomment and run after training:
# # save_model_for_streamlit(final_model, scaler, label_encoders, 
# #                          feature_names, optimal_threshold)

In [ ]:
import joblib
import json
import os

def save_model_for_deployment(model, scaler, label_encoders, feature_names, threshold=0.5):
    
    os.makedirs("deployment/model", exist_ok=True)

    # 1. Save model
    joblib.dump(model, "deployment/model/xgb_model.pkl")

    # 2. Save scaler
    joblib.dump(scaler, "deployment/model/scaler.pkl")

    # 3. Save encoders
    joblib.dump(label_encoders, "deployment/model/label_encoders.pkl")

    # 4. Save feature names
    with open("deployment/model/feature_names.json", "w") as f:
        json.dump(feature_names, f)

    # 5. Save threshold
    with open("deployment/model/threshold.json", "w") as f:
        json.dump({"threshold": threshold}, f)

    print("✅ Model saved for deployment")


# 👉 RUN THIS AFTER TRAINING
save_model_for_deployment(
    final_model,
    scaler,
    label_encoders,
    x_train.columns.tolist(),
    threshold=0.5
)